In [31]:
from pathlib import Path
import duckdb
import requests
from tqdm import tqdm
import pandas as pd

In [37]:
DATA_DIR = Path("data")
CATEGORY = "Kindle_Store"
BASE_URL = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw"
REVIEWS_URL = f"{BASE_URL}/review_categories/{CATEGORY}.jsonl.gz"
META_URL    = f"{BASE_URL}/meta_categories/meta_{CATEGORY}.jsonl.gz"
REVIEWS_FILE = DATA_DIR / f"{CATEGORY}.jsonl.gz"
META_FILE    = DATA_DIR / f"meta_{CATEGORY}.jsonl.gz"
OUTPUT_FILE  = DATA_DIR / f"{CATEGORY}_merged.parquet"

In [38]:
c2 = duckdb.connect()

In [39]:
head_reviews = c2.execute(f"SELECT * FROM read_json_auto('{REVIEWS_URL}') LIMIT 5").df()
head_reviews

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,excellent writer reminds me of Clive Cussler,GRUMLEY is on par with Clive Cussler and his D...,[],B00LXRJICK,B00LXRJICK,AFKZENTNBQ7A7V7UXW5JJI6UGRYQ,1427541413000,0,False
1,3.0,Alright book,The book Fade was not my favorite but was a go...,[],B073DFP8VC,B073DFP8VC,AHGTHCERTEZUXNBLJ5SWHK2CDLXA,1504226946142,0,True
2,5.0,Hats off to Fern Michaels for all her great ac...,I have been a fan of this author for many year...,[],B07QVH25KX,B07QVH25KX,AHFY2QSS6PK5MHSYZFI6TXUYNPLQ,1644883955777,0,True
3,5.0,Great read,This book is more than just about a dog and a ...,[],B004Y1NWQU,B004Y1NWQU,AHFY2QSS6PK5MHSYZFI6TXUYNPLQ,1363027885000,0,False
4,5.0,Add to legend f Arthur,Good twist on the ledgen of King<br />Arthur !...,[],B08M993CNC,B08M993CNC,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1637557512064,0,True


In [35]:
head_meta = c2.execute(f"SELECT * FROM read_json_auto('{META_URL}') LIMIT 5").df()
head_meta

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Buy a Kindle,The Palace (Chateau Book 4),Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.7,970,[A line was drawn in the sand.I made my choice...,[],0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Penelope Sky (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Romance]","{'Publisher': 'Hartwick Publishing (May 25, 20...",B08XPZPFY4,None
1,Buy a Kindle,Microsoft PowerPoint 2016 2013 2010 2007 Tips ...,[Print Replica] Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.3,35,"[Paperback versions are also available, includ...","[From the Author, Amelia Griggs is an Instruct...",0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Amelia Griggs (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Computers & Tech...","{'Publisher': None, 'Publication date': 'June ...",B07DH1LF1K,None
2,Buy a Kindle,Ill Wind (Anna Pigeon Mysteries Book 3),Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.4,1628,"[Lately, visitors to Mesa Verde have been brin...","[From Publishers Weekly, Barr lands another su...",7.99,[{'large': 'https://m.media-amazon.com/images/...,[],Nevada Barr (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Mystery, Thrille...",{'Publisher': 'Berkley; Reissue edition (March...,B0022Q8CTQ,None
3,Buy a Kindle,30 Healthy Easy Quick Lentil Recipes (Brad Arm...,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,3.8,47,"[If improved health you are seeking, look no f...",[],0.00,[{'large': 'https://m.media-amazon.com/images/...,[],Brad Armstrong (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Health, Fitness ...","{'Publisher': 'Brad Armstrong (March 10, 2013)...",B00BS56MLC,None
4,Buy a Kindle,The Road Home,Kindle Edition,{'avatar': 'https://m.media-amazon.com/images/...,4.5,475,"[In one of Jim Harrison’s greatest works, five...","[Review, ""A graceful novel...To read this book...",10.44,[{'large': 'https://m.media-amazon.com/images/...,[],Jim Harrison (Author) Format: Kindle Edition,"[Kindle Store, Kindle eBooks, Literature & Fic...",{'Publisher': 'Atlantic Monthly Press (Decembe...,B00155EZRS,None


In [44]:
c2.execute(f"""
    COPY (
        SELECT * FROM read_json_auto('{REVIEWS_URL}')
        LIMIT 20000
    )
    TO '../data/reviews_raw.parquet'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [45]:
c2.execute(f"""
      COPY (SELECT * FROM read_json_auto('{META_URL}') LIMIT 20000)
      TO '../data/meta_raw.parquet'
      (FORMAT PARQUET, COMPRESSION ZSTD)
  """)

In [89]:

c2.execute("""
    COPY (
        SELECT r.*, m.title AS product_title, m.price,
                    m.average_rating, m.main_category, m.store, m.description, m.features
        FROM read_parquet('../data/reviews_raw.parquet') r
        LEFT JOIN read_parquet('../data/meta_raw.parquet') m USING (parent_asin)
    )
    TO '../data/merged.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [90]:
merged_df = c2.execute(f"SELECT * FROM read_parquet('../data/merged.parquet')").df()

In [91]:
c2.execute("""
    SELECT
        rating,
        COUNT(*) AS cnt,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('../data/merged.parquet')
    GROUP BY 1
    ORDER BY 1
""").df()

,rating,cnt,pct
0,1.0,497,2.49
1,2.0,818,4.09
2,3.0,2156,10.78
3,4.0,4951,24.76
4,5.0,11578,57.89


In [92]:
merged_df.head(15)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,price,average_rating,main_category,store,description,features
0,4.0,Good start,Interesting n intertaining concept. Looking fo...,[],B07FQTND9B,B07FQTND9B,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1533236653336,0,False,Thrive: A Post-Apocalyptic Alien Survival Seri...,0.00,4.2,Buy a Kindle,Jonathan Yanez (Author) Format: Kindle Edition,"[About the Author, Jonathan Yanez is the autho...","[Would you choose peace over truth?, How long ..."
1,4.0,Some great tips on getting healthy,Some great tips on getting healthy. Great star...,[],B00QMUBB0K,B00QMUBB0K,AEINY4XOINMMJCK5GZ3M6MMHBN6A,1445287011000,0,True,Lose Weight Fast: Over 50 Incredible Weight Lo...,4.99,3.6,Buy a Kindle,"Dr. Jyothi Shenoy (Author), Lori Wenger (Edit...",[],"[Are You Sick Of Being Overweight?, This book ..."
2,5.0,Touching !,i got this book from amazon as a freebie (or m...,[],B00C8BV10W,B00C8BV10W,AHGAOIZVODNHYMNCBV4DECZH42UQ,1378581448000,1,True,A Widow Redefined,0.00,4.1,Buy a Kindle,Kim Cano (Author) Format: Kindle Edition,"[Review, Semifinalist in Literary Fiction at T...",[Amy White is one of the luckiest women alive....
3,5.0,Wow!,I absolutely loved this book! I could not put...,[],B087ZXSPGJ,B087ZXSPGJ,AEW55ZDSPC6HOXZ2RK5MCT7OM5WQ,1612906611122,0,True,The Four Winds: A Novel,11.99,4.5,Buy a Kindle,Kristin Hannah (Author) Format: Kindle Edition,"[Amazon.com Review, An Amazon Best Book of Feb...","[""The Bestselling Hardcover Novel of the Year...."
4,5.0,awesomely fun to read...,I won a copy of this book from an author event...,[],B014K1BR4C,B014K1BR4C,AFFZVSTUS3U2ZD22A2NPZSKOCPGQ,1455582556000,0,False,Switching Hour: Magic and Mayhem Book One,0.00,4.4,Buy a Kindle,Ron Peterman (Author) Format: Kindle Edition,"[Review, ""If Amy Schumer and Janet Evanovitch ...",[Released from the magic pokey and paroled wit...
5,5.0,Five well earned stars...,Loved book two. Levy grew as a character. I le...,[],B06XQ85LY5,B06XQ85LY5,AFO3G62P2JXCNMWZTAIB56KPG56A,1490902634000,0,False,Obsidian Magic (Legacy Series Book 2),0.00,4.4,Buy a Kindle,McKenzie Hunter (Author) Format: Kindle Edition,"[Review, ""This series gets, better, and, bette...",[The only thing standing between me and death ...
6,4.0,Liked it,Liked the chemistry between Luke and Reese. Pl...,[],B00ECN3JU4,B00ECN3JU4,AFO3G62P2JXCNMWZTAIB56KPG56A,1469502760000,1,True,Raising Ryann (Bad Boy Reformed 1),0.00,4.3,Buy a Kindle,Alyssa Rae Taylor (Author) Format: Kindle Ed...,"[Review, I totally didn't expect that ending b...",[A USA TODAY BESTSELLERNew Adult Romance --Rai...
7,1.0,Ummm no.,I couldn't move beyond 28% and that with me tr...,[],B016XFNFWK,B016XFNFWK,AFO3G62P2JXCNMWZTAIB56KPG56A,1467554859000,1,False,His Soul To Keep (Dark Knights of Heaven Book 1),0.00,4.5,Buy a Kindle,TW Knight (Author) Format: Kindle Edition,[],"[Where fallen angels battle Hell’s demons, fin..."
8,4.0,My OPINION of this book,"The plot, characters and story was great. I ad...",[],B00Z47DB00,B00Z47DB00,AFO3G62P2JXCNMWZTAIB56KPG56A,1464494571000,0,True,The Ascension of Laney: A Paranormal Second Ch...,0.00,3.9,Buy a Kindle,"Kris Hack (Author), Tiffany Tillman (Editor) ...","[About the Author, Kris Hack writes New Adult ...",[Standing in front of me is the man who ghoste...
9,4.0,Another great book,The book was over to quick,[],B00C74WXMK,B00C74WXMK,AF7STGSJYCAHOQAKN3JTC3JC3Y5A,1404923397000,0,True,Revealed: A House of Night Novel,9.99,4.7,Buy a Kindle,"P. C. Cast (Author), Kristin Cast (Author) ...","[From, Booklist, The situation at the Tulsa br...","[Revealed, is the spellbinding eleventh and pe..."


In [93]:
import numpy as np

def to_text(x):
    if x is pd.NA:
        return ""
    if isinstance(x, np.ndarray):
        return " ".join(map(str, x))
    return str(x)

merged_df["features"] = merged_df["features"].apply(to_text)
merged_df["description"] = merged_df["description"].apply(to_text)

In [94]:
merged_df.head(3)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,price,average_rating,main_category,store,description,features
0,4.0,Good start,Interesting n intertaining concept. Looking fo...,[],B07FQTND9B,B07FQTND9B,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1533236653336,0,False,Thrive: A Post-Apocalyptic Alien Survival Seri...,0.00,4.2,Buy a Kindle,Jonathan Yanez (Author) Format: Kindle Edition,About the Author Jonathan Yanez is the author ...,Would you choose peace over truth? How long co...
1,4.0,Some great tips on getting healthy,Some great tips on getting healthy. Great star...,[],B00QMUBB0K,B00QMUBB0K,AEINY4XOINMMJCK5GZ3M6MMHBN6A,1445287011000,0,True,Lose Weight Fast: Over 50 Incredible Weight Lo...,4.99,3.6,Buy a Kindle,"Dr. Jyothi Shenoy (Author), Lori Wenger (Edit...",,Are You Sick Of Being Overweight? This book is...
2,5.0,Touching !,i got this book from amazon as a freebie (or m...,[],B00C8BV10W,B00C8BV10W,AHGAOIZVODNHYMNCBV4DECZH42UQ,1378581448000,1,True,A Widow Redefined,0.00,4.1,Buy a Kindle,Kim Cano (Author) Format: Kindle Edition,Review Semifinalist in Literary Fiction at The...,Amy White is one of the luckiest women alive.A...


In [95]:
merged_df["document"] = (
    merged_df["product_title"].fillna("") + " " +
    merged_df["description"].fillna("") + " " +
    merged_df["features"].fillna("") + " " +
    merged_df["text"].fillna("")
)

In [96]:
merged_df.head(3)

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,price,average_rating,main_category,store,description,features,document
0,4.0,Good start,Interesting n intertaining concept. Looking fo...,[],B07FQTND9B,B07FQTND9B,AFWHJ6O3PV4JC7PVOJH6CPULO2KQ,1533236653336,0,False,Thrive: A Post-Apocalyptic Alien Survival Seri...,0.00,4.2,Buy a Kindle,Jonathan Yanez (Author) Format: Kindle Edition,About the Author Jonathan Yanez is the author ...,Would you choose peace over truth? How long co...,Thrive: A Post-Apocalyptic Alien Survival Seri...
1,4.0,Some great tips on getting healthy,Some great tips on getting healthy. Great star...,[],B00QMUBB0K,B00QMUBB0K,AEINY4XOINMMJCK5GZ3M6MMHBN6A,1445287011000,0,True,Lose Weight Fast: Over 50 Incredible Weight Lo...,4.99,3.6,Buy a Kindle,"Dr. Jyothi Shenoy (Author), Lori Wenger (Edit...",,Are You Sick Of Being Overweight? This book is...,Lose Weight Fast: Over 50 Incredible Weight Lo...
2,5.0,Touching !,i got this book from amazon as a freebie (or m...,[],B00C8BV10W,B00C8BV10W,AHGAOIZVODNHYMNCBV4DECZH42UQ,1378581448000,1,True,A Widow Redefined,0.00,4.1,Buy a Kindle,Kim Cano (Author) Format: Kindle Edition,Review Semifinalist in Literary Fiction at The...,Amy White is one of the luckiest women alive.A...,A Widow Redefined Review Semifinalist in Liter...
